# DocuRoute — train Layer 2 on Colab

Local CPU measured **~86 min per fold** at `max_length=256`, so a 5-fold
cross-validation run is over 7 hours. On a Colab T4 the same run is minutes.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Each fold is saved as soon as it finishes, so a disconnect costs you one fold,
not the whole run. The last cell zips the weights for download into
`Backend/ml/saved_models/`.

## 1. Clone the repository

In [ ]:
REPO = 'https://github.com/Doculan/DocuRoute1.git'
BRANCH = 'main'

import os, shutil
if os.path.exists('DocuRoute1'):
    shutil.rmtree('DocuRoute1')
!git clone --depth 1 --branch {BRANCH} {REPO}
%cd DocuRoute1/Backend
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install

Colab already has torch and transformers; only the few extras are needed.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 3. Dataset

`all.jsonl` and `folds.json` are committed, so the splits are rebuilt here
rather than uploaded. If `make_splits.py` does not exist yet (Phase 7), upload
`train.jsonl` / `val.jsonl` / `test.jsonl` into `ml/datasets/context_v2/`
with the file browser instead.

In [ ]:
import pathlib

splits = pathlib.Path('ml/revision_pipeline/scripts/make_splits.py')
if splits.exists():
    !python ml/revision_pipeline/scripts/make_splits.py --folds all
else:
    print('make_splits.py not present yet - upload the split files manually')

for name in ('train', 'val', 'test'):
    path = pathlib.Path(f'ml/datasets/context_v2/{name}.jsonl')
    print(f'{name:<6}', path.exists() and sum(1 for _ in path.open(encoding="utf-8")) or 'MISSING')

## 4. Train every fold

Saving happens inside `train_layer2.py` at the end of each fold, so an
interrupted run keeps whatever folds already finished. Re-run this cell and it
skips them.

In [ ]:
import json, pathlib, subprocess, time

FOLDS = list(range(5))
EPOCHS = 3
BATCH = 16          # a T4 handles 16 comfortably at max_length 384
MAX_LENGTH = 384    # no need for the CPU compromise of 256 here

folds_file = pathlib.Path('ml/datasets/context_v2/folds.json')
if not folds_file.exists():
    print('folds.json missing - running a single split instead')
    FOLDS = [None]

for fold in FOLDS:
    out_dir = f'ml/saved_models/context_v2/fold_{fold}' if fold is not None else 'ml/saved_models/context_v2'
    if pathlib.Path(out_dir, 'heads.pt').exists():
        print(f'fold {fold}: already trained, skipping')
        continue

    data_dir = f'ml/datasets/context_v2/fold_{fold}' if fold is not None else 'ml/datasets/context_v2'
    if not pathlib.Path(data_dir).exists():
        data_dir = 'ml/datasets/context_v2'

    print(f'\n=== fold {fold} ===')
    started = time.time()
    cmd = [
        'python', 'ml/revision_pipeline/scripts/train_layer2.py',
        '--data-dir', data_dir, '--out-dir', out_dir,
        '--epochs', str(EPOCHS), '--batch-size', str(BATCH),
        '--max-length', str(MAX_LENGTH), '--device', 'cuda',
    ]
    if fold is not None:
        cmd += ['--fold', str(fold)]
    subprocess.run(cmd, check=True)
    print(f'fold {fold} finished in {(time.time() - started) / 60:.1f} min')

## 5. Evaluate

In [ ]:
import pathlib

if pathlib.Path('ml/revision_pipeline/scripts/train_fusion.py').exists():
    !python ml/revision_pipeline/scripts/train_fusion.py
if pathlib.Path('ml/revision_pipeline/scripts/evaluate_pipeline.py').exists():
    !python ml/revision_pipeline/scripts/evaluate_pipeline.py --folds all

## 6. Download the weights

Unzip into `Backend/ml/saved_models/` locally. The folder is gitignored — the
encoder alone is ~254 MB per fold, well past what GitHub accepts without LFS.

`ml/reports/` is small and **is** committed, so copy those files into the repo
and commit them rather than leaving them here.

In [ ]:
import shutil, pathlib
from google.colab import files

shutil.make_archive('/content/context_v2_weights', 'zip', 'ml/saved_models/context_v2')
size_mb = pathlib.Path('/content/context_v2_weights.zip').stat().st_size / 1e6
print(f'{size_mb:.0f} MB')
files.download('/content/context_v2_weights.zip')

if pathlib.Path('ml/reports').exists():
    shutil.make_archive('/content/reports', 'zip', 'ml/reports')
    files.download('/content/reports.zip')